# Van PDF naar tabel naar grafiek
### Workshop *Onderzoek doen met Jev* · Master Human-Centered AI · Hogeschool Utrecht

Je leert het **principe**: hoe je met Jev tekst uit PDF's omzet in een tabel, en die tabel in een grafiek en een eenvoudige toets.

De casus: vijf beoordelingsrapporten van het CBG (**drie verleend, twee geweigerd**). Vraag: *geven de beoordelaars in geweigerde rapporten vaker een negatief oordeel?*

```
PDF  ──►  zinnen  ──►  Jev: 2 vragen per zin  ──►  tabel  ──►  grafiek + toets
code        code              Jev                  pandas       matplotlib / scipy
```

| Deel | Wat | Jij doet | Tijd |
|---|---|---|---|
| 0 | Opstarten | sleutel laden | 5 min |
| 1 | Kennismaken met Jev | opdracht 1: een choice- en een score-vraag | 20 min |
| 2 | PDF → zinnen | kijken | 10 min |
| 3 | Jev beoordeelt elke zin | opdracht 2: argumenten selecteren | 15 min |
| 4 | Grafiek en statistiek | opdracht 3 (grafiek) en 4 (toets) | 20 min |
| 5 | Reflectie | eigen codeboek schetsen | 10 min |

> ✏️ **Opdrachtcellen** herken je aan `# ✏️ TODO`: vervang de `...` door je eigen code. Na elke opdracht volgt een **✅ controle**. Vastgelopen? Klap de **💡 hint** open.

## 0. Opstarten

### Waarom?
Onderzoek moet je kunnen herhalen. Daarom leggen we vooraf vast met welk model we werken (een vaste versie, niet "de nieuwste") en houden we je sleutel buiten het notebook.

### Wat gaan we doen?
We installeren de pakketten, laden de sleutel uit het bestand `apikey` en maken verbinding met Jev.

- **Sleutel**: staat in het bestand `apikey` in de zip. Zet dat bestand in dezelfde map als dit notebook. In Google Colab vraagt het notebook je om het te uploaden. Zet de sleutel **nooit** in een cel en deel het bestand niet verder.
- **Modelversie**: we gebruiken vast `jev-1.13.0`. Noteer die in je methodesectie.

### Overzicht: waar staat de data?

| Wat | Waar | Soort | Gemaakt in |
|---|---|---|---|
| Je API-sleutel | bestand `apikey` | tekstbestand uit de zip | deel 0 |
| De vijf rapporten | map `pars/` | PDF-bestanden op schijf | deel 2 |
| Alle zinnen | `zinnen` | tabel (DataFrame) in het geheugen | deel 2 |
| Ruwe antwoorden van Jev | `jev_cache.json` | bestand op schijf | deel 3 |
| Zinnen + Jev-antwoorden | `zinnen` (uitgebreid) | tabel, **je dataset** | deel 3 |
| Alleen de argumenten | `argumenten` | tabel | deel 3 |
| Dataset om te bewaren | `resultaten.xlsx` | Excel-bestand op schijf | deel 4 |

Variabelen (in het geheugen) verdwijnen bij een herstart: draai dan de cellen opnieuw. Bestanden blijven staan; in Google Colab alleen zolang de sessie loopt, dus **download** wat je wilt bewaren.

**Hoe werkt de code?** Installeert de pakketten. Eén keer draaien is genoeg.

- `typesafe-sdk`: om Jev aan te roepen · `pdfplumber`: tekst uit PDF's halen · `pandas`: tabellen · `matplotlib`: grafieken · `scipy`: statistiek · `openpyxl`: Excel opslaan

In [ ]:
%pip install -q typesafe-sdk pdfplumber pandas matplotlib scipy openpyxl

**Wat betekent de output?** Hooguit installatiemeldingen. Geen foutmelding = klaar.

**Hoe werkt de code?** Instellingen en je API-sleutel. Nog geen Jev.

1. `MODEL`: de vaste Jev-versie, zodat iedereen hetzelfde model gebruikt.
2. `zoek_sleutelbestand`: zoekt het bestand `apikey` (of `apikey.txt`) naast dit notebook.
3. Staat het er niet en werk je in Google Colab, dan verschijnt er een uploadknop: kies het bestand `apikey` uit de zip.
4. De sleutel wordt in het geheugen gezet (`TYPESAFE_API_KEY`), zodat hij niet zichtbaar in het notebook komt.

In [ ]:
import os, re, json, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 110)

MODEL = "jev-1.13.0"     # vaste versie

def zoek_sleutelbestand():
    for naam in ("apikey", "apikey.txt"):
        if Path(naam).exists():
            return Path(naam)
    return None

sleutelbestand = zoek_sleutelbestand()
if sleutelbestand is None:
    try:
        from google.colab import files          # alleen in Google Colab
        print("Upload het bestand 'apikey' uit de zip:")
        for naam, inhoud in files.upload().items():
            Path(naam).write_bytes(inhoud)
        sleutelbestand = zoek_sleutelbestand()
    except ImportError:
        pass
if sleutelbestand is None:
    raise FileNotFoundError("Het bestand 'apikey' staat niet naast dit notebook. "
                            "Zet het in dezelfde map en draai deze cel opnieuw.")

os.environ["TYPESAFE_API_KEY"] = sleutelbestand.read_text().strip()
print("Sleutel geladen uit:", sleutelbestand.name)

**Wat betekent de output?** - `Sleutel geladen uit: apikey`: goed, je kunt verder.
- In Colab eerst een **uploadknop**: kies het bestand `apikey` uit de zip.
- Een `FileNotFoundError`: het bestand staat niet in dezelfde map als het notebook. Zet het erbij en draai de cel opnieuw.

**Waar staat de data?** De sleutel staat in het bestand `apikey` en, na deze cel, in het geheugen. Niet in het notebook zelf.

**Hoe werkt de code?** Maakt de verbinding met Jev: de `client`.

1. `TypeSafeClient`: de officiële Python-client van TypeSafe. Hij gebruikt de sleutel uit de vorige cel.
2. `client.models.list()`: een kleine test of de sleutel werkt. Klopt de sleutel niet, dan krijg je een duidelijke melding.

In [ ]:
from typesafe_sdk import TypeSafeClient

client = TypeSafeClient(model=MODEL, timeout=120.0)
try:
    client.models.list()
except Exception as fout:
    raise RuntimeError("Verbinding met Jev mislukt. Klopt de sleutel in 'apikey'? Melding: " + str(fout))
print("Verbonden. Model:", MODEL)

**Wat betekent de output?** - `Verbonden. Model: jev-1.13.0`: alles werkt, je kunt beginnen.
- `Verbinding met Jev mislukt`: de sleutel klopt niet of is ingetrokken, of er is geen internet. Meld het bij je docent.

**Waar staat de data?** De verbinding staat in `client`.

## 1. Kennismaken met Jev

### Waarom?
Voordat je Jev als meetinstrument gebruikt, moet je weten wat het teruggeeft en hoe je dat leest. Een getal als 0,96 lijkt precies, maar betekent iets heel anders bij een confidence van 1,0 dan bij 0,22.

### Wat gaan we doen?
Jev **schrijft geen tekst**. Je geeft een **state** (de tekst) en **vragen** met vaste antwoordmogelijkheden. Jev geeft bij elk antwoord **kansen** terug.

| Type | Je vraagt | Je krijgt terug |
|---|---|---|
| `noul` | Is dit waar? | kans 0–1 |
| `choice` | Welke optie? | optie + kans per optie + `confidence` |
| `score` | Waar op de schaal? | gewogen score + kans per niveau + `confidence` |

We stellen de vragen in het **Engels**: de rapporten zijn Engels en Engels is Jevs sterkste taal.

**Hoe werkt de code?** Je eerste vraag aan Jev.

1. `zin`: de tekst die Jev beoordeelt, de **state**.
2. `questions`: één vraag. De sleutel `"is_conclusie"` kies je zelf; het type is `noul` (waar of niet waar).
3. `client.system_one(...)`: stuurt tekst en vraag naar Jev.

In [ ]:
zin = ("Based on the submitted data for the current application, it is considered that the "
       "bridging of the product applied for to the literature is not established.")

antwoord = client.system_one(
    state=zin,
    questions={"is_conclusie": {"type": "noul",
                                "instructions": "Does this sentence state a conclusion reached by the assessors?"}},
)
print("model:", antwoord.model, "| input tokens:", antwoord.usage.input_tokens)
print("kans dat het een conclusie is:", round(antwoord.answers["is_conclusie"].noul, 2))

**Wat betekent de output?** - `model`: welke versie antwoordde. `input tokens`: hoeveel tekst Jev las (bepaalt de kosten).
- De kans: dicht bij **1** = vrijwel zeker een conclusie; dicht bij **0** = vrijwel zeker niet; rond **0,5** = Jev twijfelt.

**Waar staat de data?** In `antwoord`; de kans zelf in `antwoord.answers["is_conclusie"].noul`.

### Hoe lees je een antwoord?

Onder de sleutel die je koos, staat het antwoord: `antwoord.answers["is_conclusie"]`.

| Veld | Bij type | Betekenis |
|---|---|---|
| `.noul` | noul | Kans dat de uitspraak waar is (0–1) |
| `.choice` | choice | De optie met de hoogste kans |
| `.score` | score | Gewogen gemiddelde over de niveaus; kan ertussenin vallen |
| `.probabilities` | choice, score | Kans per optie of niveau; samen 1 |
| `.confidence` | choice, score | Hoe geconcentreerd de kansen zijn: 1 = alles op één optie, 0 = gelijk verdeeld |

⚠️ `confidence` is **niet** de hoogste kans: (p_max − 1/n) / (1 − 1/n). Drie opties, 90% op één: confidence 0,85. Een `noul` heeft geen confidence.

### ✏️ Opdracht 1: bouw een `choice`- en een `score`-vraag
1. Maak `oordeel_vraag`: een **choice** met de opties `positive`, `negative` en `neutral`. Geef elke optie een korte beschrijving.
2. Maak `stelligheid_vraag`: een **score** van 3 niveaus, van voorzichtig naar stellig.

Je `oordeel_vraag` gebruiken we in deel 3 opnieuw, voor álle zinnen.

<details><summary>💡 Hint</summary>

```python
oordeel_vraag = {"type": "choice",
                 "instructions": "What verdict does this sentence express?",
                 "criteria": {"positive": "...", "negative": "...", "neutral": "..."}}
```
Een `score` heeft `criteria` als **lijst** van niveaus, van laag naar hoog.
</details>

**Hoe werkt de code?** Twee vragen over dezelfde zin. *Jij vult de twee vragen in.*

1. `oordeel_vraag`: Jev kiest één optie. De beschrijving bij elke optie vertelt Jev wat je bedoelt.
2. `stelligheid_vraag`: Jev plaatst de zin op een schaal (een lijst, van laag naar hoog).
3. Beide vragen gaan in één keer naar Jev.

In [ ]:
# ✏️ TODO: Maak een choice-vraag met de opties positive / negative / neutral
oordeel_vraag = ...

# ✏️ TODO: Maak een score-vraag met drie niveaus, van voorzichtig naar stellig
stelligheid_vraag = ...

antwoord = client.system_one(state=zin, questions={"oordeel": oordeel_vraag, "stelligheid": stelligheid_vraag})
for k, a in antwoord.answers.items():
    print(k, {x: getattr(a, x) for x in ("choice", "score", "confidence", "probabilities") if hasattr(a, x)})

**Wat betekent de output?** Per vraag één regel. Hoe je die leest, staat hieronder.

**Waar staat de data?** Je vragen in `oordeel_vraag` en `stelligheid_vraag`; het antwoord in `antwoord`.

In [ ]:
# ✅ controle
assert oordeel_vraag["type"] == "choice" and set(oordeel_vraag["criteria"]) == {"positive", "negative", "neutral"}, \
    "oordeel_vraag: type 'choice' met precies de opties positive, negative, neutral"
assert stelligheid_vraag["type"] == "score" and isinstance(stelligheid_vraag["criteria"], list), \
    "stelligheid_vraag: type 'score' met een LIJST niveaus"
print("✅ Opdracht 1 werkt.")

### Zo lees je de uitkomst van opdracht 1

De echte uitkomst van `jev-1.13.0` voor de voorbeeldzin:

```
oordeel     {'choice': 'negative', 'confidence': 1.0, 'probabilities': {'positive': 0.0, 'negative': 1.0, 'neutral': 0.0}}
stelligheid {'score': 0.96, 'confidence': 0.22, 'probabilities': {0: 0.28, 1: 0.48, 2: 0.24}}
```

**`oordeel` (choice): Jev is zeker.** Alle kans ligt op `negative`, dus de confidence is 1,0.

**`stelligheid` (score): Jev twijfelt.** 0×0,28 + 1×0,48 + 2×0,24 = 0,96, maar de kans is over alle drie de niveaus verspreid: confidence 0,22. De sleutels zijn hier getallen (`0`, `1`, `2`).

**Kort:** zeker een negatief oordeel; hoe stellig, daar twijfelt Jev. Een score van 0,96 betekent hier dus **niet** "zeker *Qualified*", maar een gemiddelde van twijfel.

> 💡 **Les: lees een score nooit zonder de confidence.**

### Waar baseert Jev zijn antwoord op, en hoe betrouwbaar is dat?

**Waar komt het antwoord vandaan?**
1. **Alleen jouw input.** Jev ziet de state en jouw vraag met opties, verder niets: geen rest van het dossier, geen internet.
2. **Een getraind model.** TypeSafe traint Jev met wat ze *RLCD* noemen: *reinforcement learning for calibrated decisions*. Het model wordt beloond als zijn kansen overeenkomen met hoe vaak het gelijk heeft. Iedereen gebruikt hetzelfde model; je stuurt het alleen met je state en vragen, niet door het bij te trainen.
3. **Een kansverdeling.** Over jóuw opties of niveaus. Daaruit volgen `choice`, `score` en `confidence`.

**Wat we níet weten.** TypeSafe publiceert de opbouw van het model, de gewichten en de trainingsdata niet. Jev geeft ook geen redenering. Voor onderzoek is het dus een *black box*: je ziet wat erin gaat en wat eruit komt, niet waarom.

**Waarom voelt het zo betrouwbaar?** Omdat je altijd een net, getypeerd antwoord krijgt met een getal erbij, zoals confidence 1,0 bij het oordeel hierboven. Maar een getal is geen bewijs.

**Wat "gekalibreerd" wel en niet betekent.** Gekalibreerd betekent: van álle antwoorden met kans 0,8 is ongeveer 80% juist. Het is een eigenschap van een groep antwoorden. TypeSafe zegt zelf dat het *niet* garandeert dat een individueel antwoord juist is. En niemand heeft Jev getest op CBG-rapporten.

**Wat onafhankelijke tests laten zien.**

| Wat | Uitkomst | Bron |
|---|---|---|
| Standaardbenchmarks (o.a. IMDB, SST-2, HellaSwag, ARC) | 95–99% juist, kansen goed gekalibreerd | Deußer, Sparrenberg & Sifa (2026) |
| Praktijktaken (phishing, routering, injectiedetectie) | 62–87% juist; kalibratiefout (ECE) 0,12–0,14 | overzicht InfiniSynapse |
| Twee routeringstaken | 17 fouten met confidence 0,90 of hoger | overzicht InfiniSynapse |
| Volgorde van de opties omdraaien | kansen verschoven bij dezelfde input | overzicht InfiniSynapse |

Omdat de trainingsdata onbekend is, weet je ook niet of die benchmarks al in de training zaten.

**Dus: valideer zelf.** Behandel Jev als een codeur die je nog moet toetsen. Codeer een steekproef met de hand en vergelijk die met Jev (Cohen's κ, zie de reflectie in deel 5).

*Bronnen:* [TypeSafe: System One](https://docs.typesafe.ai/concepts/system-one) · [TypeSafe: AI primer](https://docs.typesafe.ai/introduction/machine-learning-primer) · [Deußer e.a., arXiv 2609.37647](https://arxiv.org/pdf/2609.37647) · [Jev Benchmark, InfiniSynapse](https://infinisynapse.com/en/blog/jev-benchmark)

## 2. Van PDF naar zinnen

### Waarom?
Jev kan geen PDF lezen en werkt het best op korte stukjes tekst. Daarom halen we eerst de tekst uit de PDF's en knippen die in zinnen. Dat is exact en herhaalbaar werk, dus doet code het, niet Jev.

### Wat gaan we doen?
We lezen de vijf PDF's, halen kop- en voetregels weg en knippen de tekst in zinnen. Het resultaat is één tabel met één rij per zin, met erbij uit welk rapport de zin komt en of dat rapport is verleend of geweigerd.

**Hoe werkt de code?** Zorgt dat de vijf PDF's in de map `pars/` staan. In Google Colab opent er een uploadknop als de map nog leeg is.

In [ ]:
DATA_DIR = Path("pars"); DATA_DIR.mkdir(exist_ok=True)
if not list(DATA_DIR.glob("*.pdf")):
    try:
        from google.colab import files
        for naam, inhoud in files.upload().items():
            (DATA_DIR / naam).write_bytes(inhoud)
    except ImportError:
        print("Zet de PDF's handmatig in de map 'pars/'.")
print(sorted(p.name for p in DATA_DIR.glob("*.pdf")))

**Wat betekent de output?** Een lijst met **vijf** bestandsnamen.

**Waar staat de data?** In de map `pars/` op schijf.

**Hoe werkt de code?** PDF → zinnen. Geen Jev; pure code.

1. `is_ruis`: herkent regels die we niet willen: logo-tekst, paginanummers, voetregels en kopjes (korte regels die met een hoofdletter of cijfer beginnen en zonder punt eindigen).
2. `lees_zinnen`: per PDF
    - leest de tekst met `pdfplumber`;
    - haalt van het titelblad de productnaam (→ korte code, bijv. `DEX`) en de uitkomst (*non-approval* → geweigerd);
    - begint bij de inleiding, stopt bij het naschrift, en knipt de tekst in zinnen (na een punt, vraagteken of dubbele punt). Zinnen korter dan 6 woorden vallen af.
3. Onderaan: alle zinnen van alle PDF's in één tabel `zinnen`.

In [ ]:
import pdfplumber

def is_ruis(r):
    return (not r or r in {"G", "B", "C", "M E B"} or re.fullmatch(r"\d+/\d+", r)
            or r.startswith("Public Assessment Report NL/H/")
            or r.lower() == "recommendation"                                           # rest van een kopje
            or (len(r.split()) <= 8 and re.match(r"[A-Z0-9]", r) and not r.endswith((".", ",", ";", ":"))))   # kopje

def lees_zinnen(pad):
    with pdfplumber.open(pad) as pdf:
        paginas = [p.extract_text() or "" for p in pdf.pages]
    titelblad = paginas[0]
    product = re.search(r"Scientific discussion\s+(.+)", titelblad).group(1).strip()
    uitkomst = "geweigerd" if "non-approval" in titelblad else "verleend"

    regels = [r.strip().lstrip("•").strip() for blad in paginas[1:] for r in blad.splitlines()]
    start = next(i for i, r in enumerate(regels) if re.match(r"(I\.\s+INTRODUCTION|1\s+Introduction)", r))
    tekst = " ".join(r for r in regels[start:] if not is_ruis(r))
    tekst = tekst.split("STEPS TAKEN AFTER")[0]                 # naschrift weglaten
    tekst = re.sub(r"-\s+(?=[a-z])", "-", tekst).replace("Ph.Eur.", "Ph Eur")
    zinnen = re.split(r"(?<=[.!?:])\s+(?=[A-Z])", tekst)                # knip na . ! ? of :
    return [{"doc_id": product[:3].upper(), "uitkomst": uitkomst, "zin": z}
            for z in zinnen if len(z.split()) >= 6]

zinnen = pd.DataFrame([rij for pad in sorted(DATA_DIR.glob("*.pdf")) for rij in lees_zinnen(pad)])
zinnen["rapport"] = zinnen.doc_id + " (" + zinnen.uitkomst + ")"
print(zinnen.rapport.value_counts().to_string())
zinnen.sample(5, random_state=1)

**Wat betekent de output?** - Bovenaan: **aantal zinnen per rapport**. De goedgekeurde rapporten hebben er veel meer dan de geweigerde: een afwijzing beschrijft alleen wat misging. Daarom vergelijken we straks **percentages**, geen aantallen.
- Daaronder: vijf willekeurige zinnen. Het knippen is niet perfect (soms plakt er een stukje kop aan een zin); voor het principe is dat genoeg.

**Waar staat de data?** In de tabel `zinnen`: per zin `doc_id`, `uitkomst`, `rapport` en de `zin` zelf.

## 3. Jev beoordeelt elke zin

### Waarom?
Hier wordt tekst data: per zin stelt Jev dezelfde vragen, en elk antwoord wordt een kolom. We houden het bij **twee** vragen, zodat je goed ziet wat er gebeurt. We vragen het **per zin** en niet per heel rapport: zo krijg je één rij per zin, werkt Jev nauwkeuriger op weinig tekst, en kun je elk antwoord terugvinden in de bron.

### Wat gaan we doen?
Het codeboek heeft twee vragen:
1. `is_argument` (noul): bevat de zin een oordeel, reden of besluit, of is het alleen beschrijving?
2. `oordeel` (choice): **jouw** `oordeel_vraag` uit opdracht 1: positief, negatief of neutraal?

Daarna laat je Jev alle zinnen beoordelen en selecteer je de argumenten.

**Hoe werkt de code?** Het codeboek: de twee vragen die Jev over **elke** zin stelt. `oordeel` is precies jouw vraag uit opdracht 1. Er wordt nog niets naar Jev gestuurd.

In [ ]:
CODEBOEK = {
    "is_argument": {
        "type": "noul",
        "instructions": ("Does this sentence contain a judgement, reason, requirement or decision "
                         "about the marketing authorisation, rather than only a description?"),
        "criteria": {"true": "It evaluates, justifies, requires, concludes or decides something",
                     "false": "It only describes the product, the procedure or the composition"},
    },
    "oordeel": oordeel_vraag,        # jouw vraag uit opdracht 1
}
print("Vragen in het codeboek:", list(CODEBOEK))

**Wat betekent de output?** `Vragen in het codeboek: ['is_argument', 'oordeel']`. Elke vraag wordt zo meteen een kolom.

**Waar staat de data?** In `CODEBOEK`.

**Hoe werkt de code?** Jev beoordeelt alle zinnen. Dit is de stap van tekst naar tabel.

1. `vraag_jev`: stuurt één zin met beide vragen naar Jev en haalt de waarden eruit: de kans op `is_argument`, en het `oordeel` met zijn `oordeel_conf`.
2. Elk antwoord wordt bewaard in `jev_cache.json`. Draai je opnieuw, dan komt het daar vandaan: gratis en hetzelfde.
3. `ThreadPoolExecutor`: 8 zinnen tegelijk, zodat het sneller gaat (ongeveer een minuut).
4. De antwoorden komen als kolommen naast de zinnen in de tabel `zinnen`.

In [ ]:
CACHE = Path("jev_cache.json")
cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}

def vraag_jev(zin):
    sleutel = hashlib.sha256(json.dumps([MODEL, zin, CODEBOEK], sort_keys=True).encode()).hexdigest()
    if sleutel not in cache:
        a = client.system_one(state=zin, questions=CODEBOEK).answers
        cache[sleutel] = {"is_argument": a["is_argument"].noul,
                          "oordeel": a["oordeel"].choice,
                          "oordeel_conf": a["oordeel"].confidence}
    return cache[sleutel]

with ThreadPoolExecutor(max_workers=8) as pool:
    antwoorden = list(pool.map(vraag_jev, zinnen.zin))
CACHE.write_text(json.dumps(cache))

zinnen = zinnen[["doc_id", "uitkomst", "rapport", "zin"]].join(pd.DataFrame(antwoorden))
zinnen.head(10)

**Wat betekent de output?** De eerste tien rijen van je dataset. Per zin:
- `is_argument`: kans 0–1 dat er een argument in zit (dicht bij 1 = ja).
- `oordeel`: positive, negative of neutral.
- `oordeel_conf`: hoe zeker Jev over dat oordeel is (1 = zeker, 0 = gelijk verdeeld).

**Waar staat de data?** In de tabel `zinnen` (**je dataset**) en, als ruwe antwoorden, in `jev_cache.json` op schijf.

**Hoe werkt de code?** Toont de vijf zinnen waar Jev het minst zeker was over het oordeel (laagste `oordeel_conf` bovenaan).

In [ ]:
zinnen.sort_values("oordeel_conf")[["rapport", "zin", "oordeel", "oordeel_conf"]].head(5)

**Wat betekent de output?** Hier twijfelde Jev tussen oordelen. Lees de zinnen: twijfel jij ook? Vaak zijn het zinnen die echt dubbelzinnig zijn, of waar de opties in je `oordeel_vraag` op elkaar lijken. Dat zijn de zinnen om met de hand na te kijken.

### ✏️ Opdracht 2: selecteer de argumenten
Niet elke zin is een argument; veel zinnen beschrijven alleen het product of de procedure. Maak `argumenten`: alleen de rijen van `zinnen` met `is_argument` **van 0,5 of hoger**.

<details><summary>💡 Hint</summary>

```python
argumenten = zinnen[zinnen.is_argument >= 0.5]
```
</details>

**Hoe werkt de code?** Filtert de tabel. *Jij schrijft de filterregel.* Daarna telt de cel hoeveel argumenten er per rapport zijn.

In [ ]:
# ✏️ TODO: Houd alleen de zinnen over met is_argument >= 0.5
argumenten = ...

print(f"{len(argumenten)} van de {len(zinnen)} zinnen bevatten volgens Jev een argument\n")
print(argumenten.rapport.value_counts().to_string())

**Wat betekent de output?** Hoeveel zinnen een argument zijn, in totaal en per rapport. Je ziet dat lang niet elke zin een argument is: een groot deel van een rapport is beschrijving.

**Waar staat de data?** In de tabel `argumenten`.

In [ ]:
# ✅ controle
assert isinstance(argumenten, pd.DataFrame) and len(argumenten) > 0, "argumenten moet een (niet-lege) tabel zijn"
assert (argumenten.is_argument >= 0.5).all(), "er zitten nog zinnen in met is_argument < 0.5"
assert len(argumenten) == (zinnen.is_argument >= 0.5).sum(), "er ontbreken zinnen met is_argument >= 0.5"
print("✅ Opdracht 2 werkt.")

## 4. Grafiek en statistiek

### Waarom?
Nu de tekst een tabel is, kun je tellen en vergelijken. Een grafiek laat in één oogopslag zien of de rapporten verschillen; een eenvoudige toets laat zien of dat verschil groot genoeg is om serieus te nemen, en waar de grenzen liggen.

### Wat gaan we doen?
1. Opdracht 3: per rapport het percentage positieve, negatieve en neutrale argumenten, in een grafiek.
2. Opdracht 4: verleend tegen geweigerd, met een χ²-toets.
3. De dataset opslaan in Excel.

### ✏️ Opdracht 3: grafiek per rapport
Maak `tab`: een kruistabel met `rapport` als rijen en `oordeel` als kolommen, **in procenten per rapport**. De grafiek wordt daarna vanzelf getekend.

<details><summary>💡 Hint</summary>

```python
tab = pd.crosstab(argumenten.rapport, argumenten.oordeel, normalize="index") * 100
```
`normalize="index"` maakt van elke rij percentages die samen 100 zijn.
</details>

**Hoe werkt de code?** *Jij maakt de kruistabel.* Daarna tekent de cel een liggende staafgrafiek: per rapport één balk, opgedeeld in positief (blauw), negatief (rood) en neutraal (grijs).

In [ ]:
# ✏️ TODO: Maak een kruistabel rapport × oordeel, in procenten per rapport
tab = ...

# volgorde: geweigerde rapporten bovenaan, kleuren van positief via neutraal naar negatief
tab = tab.loc[sorted(tab.index, key=lambda r: ("geweigerd" in r, r))]
tab = tab[[c for c in ["positive", "neutral", "negative"] if c in tab.columns]]
kleuren = {"positive": "#00A0D2", "negative": "#EC0010", "neutral": "#BBBBBB"}
ax = tab.plot(kind="barh", stacked=True, figsize=(9, 3.5), color=[kleuren.get(c, "#888888") for c in tab.columns])
ax.set_xlabel("% van de argumenten"); ax.set_ylabel(""); ax.set_title("Oordeel per rapport, volgens Jev")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); plt.show()
tab.round(0)

**Wat betekent de output?** Per rapport hoe de argumenten verdeeld zijn over positief, negatief en neutraal. Kijk vooral naar de **rode** delen: hebben de twee geweigerde rapporten (DEX, GLY) een groter negatief deel dan de drie verleende? Onder de grafiek staan dezelfde percentages als getallen.

**Waar staat de data?** In `tab`.

In [ ]:
# ✅ controle
assert isinstance(tab, pd.DataFrame) and set(tab.index) == set(argumenten.rapport), "tab: rapporten als rijen"
assert np.allclose(tab.sum(axis=1), 100), "elke rij moet samen 100% zijn: gebruik normalize='index' en * 100"
print("✅ Opdracht 3 werkt.")

### ✏️ Opdracht 4: een statistiekje
Is het verschil tussen verleende en geweigerde rapporten groot genoeg om serieus te nemen?
1. Maak `tab2`: een kruistabel met `uitkomst` als rijen en `oordeel` als kolommen, nu met **aantallen** (geen `normalize`).
2. Voer de χ²-toets uit: `chi2, p, dof, verwacht = chi2_contingency(tab2)`.

<details><summary>💡 Hint</summary>

```python
tab2 = pd.crosstab(argumenten.uitkomst, argumenten.oordeel)
chi2, p, dof, verwacht = chi2_contingency(tab2)
```
</details>

**Hoe werkt de code?** *Jij maakt de tabel en roept de toets aan.* De χ²-toets vergelijkt de aantallen die je ziet met de aantallen die je zou verwachten als het oordeel **niets** met de uitkomst te maken had.

In [ ]:
from scipy.stats import chi2_contingency

# ✏️ TODO: Tel oordeel per uitkomst en voer de χ²-toets uit
tab2 = ...
chi2, p, dof, verwacht = ...

print(f"χ² = {chi2:.1f}, df = {dof}, p = {p:.3f}")
tab2

**Wat betekent de output?** - De tabel: hoeveel positieve, negatieve en neutrale argumenten er zijn bij verleende en bij geweigerde rapporten.
- `p`: de kans op een verschil van deze omvang als oordeel en uitkomst niets met elkaar te maken hebben. Is `p` kleiner dan 0,05, dan noemen we het verschil meestal *significant*.

⚠️ **Maar pas op.** Zinnen uit hetzelfde rapport zijn niet onafhankelijk, en we hebben maar vijf rapporten. Een kleine p zegt hier dus vooral iets over déze vijf rapporten, niet over CBG-rapporten in het algemeen. Bovendien zijn de oordelen door Jev gegeven, niet door mensen gecontroleerd.

**Waar staat de data?** In `tab2`, `chi2` en `p`.

In [ ]:
# ✅ controle
assert isinstance(tab2, pd.DataFrame) and set(tab2.index) <= {"verleend", "geweigerd"}, "tab2: uitkomst als rijen"
assert tab2.values.sum() == len(argumenten), "tab2 moet aantallen bevatten (geen normalize)"
assert 0 <= p <= 1
print("✅ Opdracht 4 werkt.")

**Hoe werkt de code?** Slaat je dataset op als Excel-bestand, zodat je hem kunt bewaren of in een ander programma kunt openen.

In [ ]:
zinnen.to_excel("resultaten.xlsx", index=False)
print("Opgeslagen:", len(zinnen), "zinnen in resultaten.xlsx")

**Wat betekent de output?** **Waar staat de data?** In `resultaten.xlsx` op schijf: per zin het rapport, de uitkomst, de zin en de antwoorden van Jev. In Colab: download het via het mapje links.

## 5. Reflectie en jouw eigen onderzoek

### Waarom?
De casus is een oefening; het gaat om wat je meeneemt naar je eigen onderzoek: hoe je met een paar goed gekozen vragen tekst omzet in data, en hoe kritisch je daarna naar die data moet kijken.

### Wat gaan we doen?
Bespreek in je groep en noteer kort:

1. **Meting.** Je `oordeel_vraag` bepaalde de hele grafiek. Wat gebeurt er als je de beschrijving van de opties verandert?
2. **Vertrouwen.** Jev gaf vaak een confidence van 1,0. Hoe weet je of dat terecht is? *(Tip: codeer zelf 20 zinnen en vergelijk.)*
3. **Validiteit.** Vijf rapporten, twee weigeringen. Wat kun je wél concluderen?
4. **Overdracht.** Welke tekst uit je **eigen** onderzoek (interviews, beleidsstukken, reviews, chatlogs) kun je zo in een tabel omzetten? Schets hieronder 2–3 vragen met type en opties.

**Hoe werkt de code?** Niets automatisch: een lege schets voor het codeboek van je eigen onderzoek.

In [ ]:
# ✏️ Mijn codeboek voor mijn eigen onderzoek (schets)
MIJN_CODEBOEK = {
    # "vraag_1": {"type": "noul",   "instructions": "..."},
    # "vraag_2": {"type": "choice", "instructions": "...", "criteria": {"...": "...", "...": "..."}},
    # "vraag_3": {"type": "score",  "instructions": "...", "criteria": ["...", "...", "..."]},
}